# Sparse-aperture interferometric reconstruction with a CNN-Transformer

End-to-end walkthrough of the prototype. With the fast defaults below, expect ~5–10 minutes on CPU or ~15 minutes on a Colab T4.

**Pipeline**
1. Generate (or load) a set of ground-truth galaxy images.
2. Forward-model each through a simulated ALMA-like sparse uv-coverage to obtain dirty images.
3. Train a U-Net with a Transformer self-attention bottleneck to reconstruct truth from dirty.
4. Compare against the classical Hogbom CLEAN baseline using PSNR, SSIM and peak-position error.

**Author** Pratyush Kumar Das — University of Queensland. Built as a proof-of-concept toward ML-based imaging for ALMA / SKA.

## 0. Setup

In [ ]:
!pip install "numpy>=1.24,<2" torch torchvision scipy scikit-image matplotlib astropy h5py tqdm

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "train.py").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "src" / "train.py").exists():
            ROOT = parent
            break
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
for d in ("data", "checkpoints/colab", "results"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT = ROOT / "checkpoints/colab/best.pt"
print("cwd:", ROOT)
print("device:", device)

## 1. The forward model: from sky image to dirty image

An interferometer measures the visibilities $V(u, v) = \mathcal{F}\{I(l, m)\}$ on a *finite, sparse* set of $(u, v)$ points. Inverse-FFT of those samples gives the dirty image:

$$ I_d(l, m) = \mathcal{F}^{-1}\{M(u, v) \cdot V(u, v)\}, $$

convolved with the dirty beam $B = \mathcal{F}^{-1}\{M\}$. The reconstruction task is to invert this band-limited, non-uniform sampling.

In [ ]:
from src.uv_simulator import make_uv_mask, dirty_image

# build a synthetic galaxy and visualise the forward model
n = 256
yy, xx = np.mgrid[:n, :n]
img = np.zeros((n, n))
for cy, cx, amp, sig in [(128, 128, 1.0, 14.0), (110, 150, 0.4, 6.0)]:
    img += amp * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sig ** 2))
img /= img.max()

mask = make_uv_mask(n_pix=n, n_ant=24, declination_deg=-30.0,
                    hour_angle_range_h=(-2, 2), n_times=80)
I_d, beam = dirty_image(img, mask, noise_sigma=0.005)

fig, ax = plt.subplots(1, 4, figsize=(13, 3.4), constrained_layout=True)
for a, im, t in zip(ax,
                    [img, mask.astype(float), I_d, beam],
                    ['truth', 'uv-mask', 'dirty image', 'dirty beam']):
    a.imshow(im, origin='lower', cmap='magma'); a.set_title(t)
    a.set_xticks([]); a.set_yticks([])
print(f'uv-mask fill fraction: {mask.mean()*100:.2f}%')
plt.show()

## 2. Build a training set

We use 1000 synthetic Sersic-profile galaxies (cheap, perfect ground truth). For real-data experiments, swap `--source galaxy10` in `scripts/download_data.py`.

In [ ]:
!python scripts/download_data.py --source synthetic --n 400 --size 128 --out data/galaxies.h5

## 3. Train

**Fast defaults** (`--fast`, 128 px, subset of galaxies): ~5–10 min on CPU, ~12 min on Colab T4. For full quality use `--size 256 --n 1000` and drop `--fast --max-samples`.

In [ ]:
import torch
_fast = "--fast --max-samples 400"
if torch.cuda.is_available():
    _train = f"--epochs 30 --batch-size 16 --base 16 --heads 4 {_fast}"
else:
    _train = f"--epochs 8 --batch-size 8 --base 8 --heads 4 {_fast}"

if CKPT.is_file():
    print(f"Using existing checkpoint ({CKPT.stat().st_size // 1024} KiB). Delete it to retrain.")
else:
    get_ipython().system(
        f"cd src && python train.py --data ../data/galaxies.h5 --out ../checkpoints/colab "
        f"{_train} --lambda-ssim 0.2 --lr 2e-4 --workers 0"
    )
    if not CKPT.is_file():
        raise FileNotFoundError(f"Training did not produce {CKPT}")

## 4. Evaluate against CLEAN

PSNR / SSIM / peak-position error on 16 held-out galaxies. Hogbom CLEAN uses 100 iterations by default (`--clean-iters 400` for publication quality).

In [ ]:
if not CKPT.is_file():
    raise FileNotFoundError(f"Missing {CKPT}; run §3 (Train) first.")
get_ipython().system(
    "cd src && python eval.py --data ../data/galaxies.h5 "
    "--checkpoint ../checkpoints/colab/best.pt "
    "--n-test 16 --fast --clean-iters 100 --out ../results/eval_colab.json"
)

## 5. Headline figure

In [ ]:
from IPython.display import Image, display

if not CKPT.is_file():
    raise FileNotFoundError(f"Missing {CKPT}; run §3 (Train) first.")

out = ROOT / "results/headline.png"
get_ipython().system(
    "python scripts/make_results_figure.py "
    "--data data/galaxies.h5 "
    "--checkpoint checkpoints/colab/best.pt "
    "--fast --clean-iter 100 "
    f"--out {out}"
)
if out.is_file():
    display(Image(filename=str(out)))
else:
    print(f"Figure not found: {out}")

## 6. Where this would go next

- **Real visibilities.** Drop in CASA-simulated visibilities of ALMA archival fields and retrain on (dirty, CLEAN-restored) pairs.
- **Visual Transformer.** Replace the single bottleneck attention block with a stack of ViT blocks operating on patchified visibility space (closer to the ALMA BRAIN line of work).
- **Self-supervised pretraining.** Mask additional uv-cells at training time and require the network to reconstruct the held-out visibilities (a Fourier-domain analogue of MAE).
- **Metadata-aware conditioning.** Concatenate the dirty beam, the uv-mask, and antenna-configuration metadata as additional input channels so the same network generalises across array configurations.
- **Anomaly head.** Add an auxiliary head that scores per-pixel reconstruction confidence, enabling automatic flagging of unusual sources downstream.